# Phase 2 — Compiler-feedback loop (self-compiling)
Takes a model's round-0 **validated** patches, and for each failing patch feeds the compiler error back for N rounds. Works for base or instruct (chat auto). Upload `test_functions.csv` and the round-0 `patches_validated_<alias>_512.jsonl` as a Kaggle dataset. In-loop compile uses Kaggle's gcc; re-validate finals locally on GCC 15 for headline-comparable numbers.

In [ ]:
# ── CONFIG — edit this cell ─────────────────────────────────────────────
# Phase 2: compiler-feedback loop. Point MODEL_NAME at the SAME model whose
# round-0 validated file you upload below.
MODEL_NAME  = 'deepseek-ai/deepseek-coder-1.3b-instruct'   # or ..-1.3b-base
MODEL_ALIAS = 'deepseek_1.3b_instruct'                     # or deepseek_1.3b

N_ROUNDS          = 3
PROMPT_TYPE       = 'zero_shot'   # feedback loop operates on this subset
SAMPLE_IDX        = 0             # one round-0 candidate per function
MAX_NEW_TOKENS    = 512
TEMPERATURE       = 0.8
MAX_PROMPT_TOKENS = 1600
HF_TOKEN          = ''

USE_CHAT = 'instruct' in MODEL_NAME.lower()
OUTPUT_FILE = f'/kaggle/working/patches_feedback_{MODEL_ALIAS}.jsonl'
print(f'Feedback loop: {MODEL_ALIAS} | chat={USE_CHAT} | {N_ROUNDS} rounds | {PROMPT_TYPE} s{SAMPLE_IDX}')

In [ ]:
!pip install -q transformers accelerate

In [ ]:
import torch
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('GPU' if device.type == 'cuda' else 'NO GPU — enable in Settings -> Accelerator')

In [ ]:
import glob, json
# Round-0 comes from a VALIDATED jsonl (has func_before/patch) uploaded as a dataset.
vals = glob.glob(f'/kaggle/input/**/*validated*{MODEL_ALIAS}*.jsonl', recursive=True) \
       or glob.glob('/kaggle/input/**/*validated*.jsonl', recursive=True)
assert vals, 'Upload the round-0 validated jsonl for this model as a dataset.'
INPUT_PATH = vals[0]
print('round-0 input:', INPUT_PATH)

seed = {}
for line in open(INPUT_PATH, encoding='utf-8'):
    line = line.strip()
    if not line:
        continue
    r = json.loads(line)
    if r.get('prompt_type') == PROMPT_TYPE and r.get('sample_idx') == SAMPLE_IDX:
        seed[r['func_id']] = {'func_id': r['func_id'], 'cwe_id': r['cwe_id'],
                              'func_before': r['func_before'], 'func_after': r['func_after'],
                              'patch': r.get('patch', '')}
print(f'{len(seed)} round-0 candidates ({PROMPT_TYPE}, sample {SAMPLE_IDX})')

In [ ]:
import re

def core_zero_shot(fb, cwe):
    return (f'The following C/C++ function contains a {cwe} vulnerability.\n'
            f'Rewrite the function to fix the vulnerability.\n'
            f'Return ONLY the fixed function inside a ```c code block, no explanation.\n\n'
            f'Vulnerable function:\n```c\n{fb}\n```')

def core_few_shot(fb, cwe, pool):
    ex = pool.get(cwe, [])[:2]
    if not ex:
        return core_zero_shot(fb, cwe)
    s = f'Here are examples of fixing {cwe} vulnerabilities in C/C++.\n\n'
    for i, e in enumerate(ex, 1):
        s += (f'### Example {i}\nVulnerable:\n```c\n{e["func_before"]}\n```\n'
              f'Fixed:\n```c\n{e["func_after"]}\n```\n\n')
    s += (f'### Now fix this function\nVulnerable:\n```c\n{fb}\n```\n'
          f'Return ONLY the fixed function inside a ```c code block.')
    return s

def core_cot(fb, cwe):
    return (f'The C/C++ function below contains a {cwe} vulnerability.\n'
            f'Identify where the vulnerability occurs, what causes it, and what change fixes it.\n'
            f'Then write the complete corrected function inside a ```c code block.\n\n'
            f'Vulnerable function:\n```c\n{fb}\n```')

def extract_code(text):
    """Pull the fixed function out of a model response (chat or completion)."""
    m = re.search(r'```(?:c|cpp|c\+\+)?\s*\n(.*?)```', text, re.DOTALL)
    if m and m.group(1).strip():
        return m.group(1).strip()
    m2 = re.search(r'^(.*?)```', text, re.DOTALL)   # completion: code before first fence
    if m2 and m2.group(1).strip():
        return m2.group(1).strip()
    return text.strip()

print('Prompt builders loaded. Chat mode =', USE_CHAT)


In [ ]:
# Portable stub-injection compile harness (ported from src/validation/compile_check.py,
# Linux/Kaggle gcc; variadic function stubs + name reconciliation as in Exp 3d).
import subprocess, tempfile, os, re as _re

GCC_PATH, GPP_PATH = 'gcc', 'g++'
MAX_STUB_ITERATIONS = 6
_HEADER_PREFIX = """\
#include <stdint.h>
#include <stdbool.h>
#include <stddef.h>
#include <stdlib.h>
#include <string.h>
#include <stdio.h>
typedef uint8_t  u8;  typedef uint8_t  __u8;
typedef uint16_t u16; typedef uint16_t __u16; typedef uint16_t __le16; typedef uint16_t __be16;
typedef uint32_t u32; typedef uint32_t __u32; typedef uint32_t __le32; typedef uint32_t __be32;
typedef uint64_t u64; typedef uint64_t __u64; typedef uint64_t __le64; typedef uint64_t __be64;
typedef int8_t   s8;  typedef int16_t s16; typedef int32_t s32; typedef int64_t s64;
typedef unsigned int  gfp_t;
typedef unsigned long ulong;
typedef unsigned char uchar;
typedef unsigned char u_char;
typedef unsigned char byte;
typedef unsigned long usec_t;
typedef int           BOOL;
typedef int           status_t;
typedef uint32_t      quint32;
typedef uint64_t      quint64;
typedef int32_t       ogg_int32_t;
typedef int64_t       ogg_int64_t;
#define BGD_DECLARE(x) x

"""
_CPP_MARKERS = ['::', 'template<', 'template <', 'nullptr', 'override',
                'public:', 'private:', 'protected:', 'virtual ']
_RESERVED = {
    'int','char','void','short','long','float','double','signed','unsigned',
    'const','volatile','static','extern','auto','register','inline',
    'struct','union','enum','typedef','sizeof','return','if','else','for',
    'while','do','switch','case','default','break','continue','goto','this',
    'true','false','NULL','nullptr','size_t','ssize_t','ptrdiff_t',
    'uint8_t','uint16_t','uint32_t','uint64_t','int8_t','int16_t','int32_t','int64_t',
    'FILE','u8','u16','u32','u64','s8','s16','s32','s64',
}
_PAT_TYPE      = _re.compile(r"unknown type name '([A-Za-z_]\w*)'")
_PAT_TYPE_CPP  = _re.compile(r"'([A-Za-z_]\w*)' does not name a type")
_PAT_FUNC      = _re.compile(r"implicit declaration of function '([A-Za-z_]\w*)'")
_PAT_UNDECL    = _re.compile(r"'([A-Za-z_]\w*)' undeclared")
_PAT_STRUCT    = _re.compile(r"incomplete type 'struct ([A-Za-z_]\w*)'")
_PAT_STRUCT2   = _re.compile(r"dereferencing pointer to incomplete type 'struct ([A-Za-z_]\w*)'")
_PAT_FIELD     = _re.compile(r"(?:request for member|has no member named) '([A-Za-z_]\w*)'")
_PAT_NOT_DECL_SCOPE = _re.compile(r"'([A-Za-z_]\w*)' was not declared in this scope")

def clean_patch(code):
    code = _re.sub(r'^```(?:c|cpp|c\+\+)?\s*\n?', '', code.strip(), flags=_re.MULTILINE)
    code = _re.sub(r'\n?```\s*$', '', code.strip(), flags=_re.MULTILINE)
    return code.strip()

def _looks_like_cpp(code):
    return any(m in code for m in _CPP_MARKERS)

def _update_state(stderr, state):
    before = {k: len(v) for k, v in state.items()}
    for name in _PAT_FIELD.findall(stderr):
        if name not in _RESERVED: state['fields'].add(name)
    for name in _PAT_STRUCT.findall(stderr) + _PAT_STRUCT2.findall(stderr):
        state['structs'].add(name)
    for name in _PAT_TYPE.findall(stderr):
        if name not in _RESERVED: state['types'].add(name)
    for name in _PAT_TYPE_CPP.findall(stderr):
        if name not in _RESERVED: state['types'].add(name)
    for name in _PAT_FUNC.findall(stderr):
        if name not in _RESERVED and name not in state['vars']: state['funcs'].add(name)
    for name in _PAT_UNDECL.findall(stderr):
        if name not in _RESERVED and name not in state['funcs']: state['vars'].add(name)
    for name in _PAT_NOT_DECL_SCOPE.findall(stderr):
        if name not in _RESERVED and name not in state['funcs'] and name not in state['types']:
            state['vars'].add(name)
    return any(len(state[k]) > before[k] for k in state)

def _build_stubs(state):
    lines = []
    types = set(state['types'])
    funcs = set(state['funcs']) - types
    vars_ = set(state['vars']) - types - funcs
    if state['fields']:
        field_decls = ' '.join(f'long long {f};' for f in sorted(state['fields']))
    else:
        field_decls = 'int _pad;'
    lines.append(f'struct __field_holder {{ {field_decls} }};')
    for name in sorted(state['structs']):
        lines.append(f'struct {name} {{ int _pad[64]; }};')
    for name in sorted(types):
        lines.append(f'typedef struct __field_holder* {name};')
    for name in sorted(funcs):
        lines.append(f'int {name}(...);')
    for name in sorted(vars_):
        lines.append(f'int {name};')
    return '\n'.join(lines) + '\n'

def _run_compiler(compiler, lang, filepath):
    return subprocess.run([compiler, '-fsyntax-only', '-w', '-x', lang, filepath],
                          capture_output=True, text=True, timeout=15)

def _compile_with_stubs(code, is_cpp):
    compiler = GPP_PATH if is_cpp else GCC_PATH
    lang     = 'c++'    if is_cpp else 'c'
    suffix   = '.cpp'   if is_cpp else '.c'
    state = {'types': set(), 'structs': set(), 'funcs': set(), 'vars': set(), 'fields': set()}
    last_err = ''
    for _ in range(MAX_STUB_ITERATIONS):
        stubs = _build_stubs(state)
        tmp = tempfile.NamedTemporaryFile(suffix=suffix, mode='w', delete=False,
                                          encoding='utf-8', newline='\n')
        tmp.write(stubs + _HEADER_PREFIX + code); tmp.close()
        try:
            result = _run_compiler(compiler, lang, tmp.name)
        finally:
            os.unlink(tmp.name)
        if result.returncode == 0:
            return True, ''
        last_err = result.stderr
        if not _update_state(result.stderr, state):
            break
    return False, last_err

def is_compilable(code):
    code = clean_patch(code)
    if len(code.strip()) < 10:
        return {'compilable': False, 'error': 'Empty or too short'}
    is_cpp = _looks_like_cpp(code)
    try:
        ok, err = _compile_with_stubs(code, is_cpp)
        if ok:
            return {'compilable': True, 'error': None}
        if not is_cpp and ('::' in code or '::' in err):
            ok2, _ = _compile_with_stubs(code, is_cpp=True)
            if ok2:
                return {'compilable': True, 'error': None}
        return {'compilable': False, 'error': (err or '')[:300].strip()}
    except subprocess.TimeoutExpired:
        return {'compilable': False, 'error': 'Timeout'}

import subprocess as _sp
_v = _sp.run(['gcc', '--version'], capture_output=True, text=True).stdout.splitlines()[0]
print('Compile harness ready. Kaggle', _v)


In [ ]:
from transformers import AutoTokenizer, AutoModelForCausalLM

print(f'Loading {MODEL_NAME} ...')
load_kwargs = dict(torch_dtype=torch.float16, device_map='auto')
tok_kwargs  = {}
if HF_TOKEN:
    load_kwargs['token'] = HF_TOKEN; tok_kwargs['token'] = HF_TOKEN

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME, **tok_kwargs)
model     = AutoModelForCausalLM.from_pretrained(MODEL_NAME, **load_kwargs)
model.eval()
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token
tokenizer.truncation_side = 'left'

if USE_CHAT and tokenizer.chat_template is None:
    raise RuntimeError('USE_CHAT is True but this tokenizer has no chat template.')
print(f'Model loaded on {next(model.parameters()).device}  | chat mode = {USE_CHAT}')


In [ ]:
def generate_one(core_text):
    """Generate a completion for one prompt core, honouring chat vs completion mode.
    Renders the chat template to a STRING first (robust across transformers versions),
    then tokenises; add_special_tokens=False for chat avoids a double BOS."""
    if USE_CHAT:
        text = tokenizer.apply_chat_template(
            [{'role': 'user', 'content': core_text}],
            tokenize=False, add_generation_prompt=True)
        enc = tokenizer(text, return_tensors='pt', truncation=True,
                        max_length=MAX_PROMPT_TOKENS, add_special_tokens=False).to(device)
    else:
        enc = tokenizer(core_text + '\n\nFixed function:\n```c\n',
                        return_tensors='pt', truncation=True,
                        max_length=MAX_PROMPT_TOKENS).to(device)
    input_len = enc['input_ids'].shape[1]
    with torch.no_grad():
        out = model.generate(**enc, max_new_tokens=MAX_NEW_TOKENS, do_sample=True,
                             temperature=TEMPERATURE, top_p=0.95,
                             pad_token_id=tokenizer.eos_token_id)
    return tokenizer.decode(out[0][input_len:], skip_special_tokens=True)


In [ ]:
def feedback_core(fb, prev, err, cwe):
    err = (err or '')[:500]
    return (f'A previous attempt to fix a {cwe} vulnerability in this C/C++ function '
            f'does not compile.\n'
            f'Vulnerable function:\n```c\n{fb}\n```\n'
            f'Previous attempt:\n```c\n{prev}\n```\n'
            f'Compiler error:\n{err}\n'
            f'Fix the compiler error and return ONLY the corrected function inside a '
            f'```c code block.')

In [ ]:
import json, time
# Establish round-0 compile status under THIS toolchain (Kaggle gcc) for consistency.
state = {}
for fid, c in seed.items():
    r = is_compilable(c['patch'])
    state[fid] = {**c, 'compilable': r['compilable'], 'compile_error': r['error']}

def dump_round(rnd):
    with open(OUTPUT_FILE, 'a', encoding='utf-8') as out:
        for fid, c in state.items():
            out.write(json.dumps({
                'model_alias': MODEL_ALIAS, 'func_id': fid, 'cwe_id': c['cwe_id'],
                'prompt_type': PROMPT_TYPE, 'round': rnd, 'patch': c['patch'],
                'compilable': c['compilable'], 'compile_error': c['compile_error'],
                'func_before': c['func_before'], 'func_after': c['func_after'],
            }, ensure_ascii=False) + '\n')

open(OUTPUT_FILE, 'w').close()      # fresh
dump_round(0)
n0 = sum(c['compilable'] for c in state.values())
print(f'round 0: {n0}/{len(state)} compile ({n0/len(state)*100:.1f}%)')

for rnd in range(1, N_ROUNDS + 1):
    fails = [fid for fid, c in state.items() if not c['compilable']]
    print(f'\n--- round {rnd}: retrying {len(fails)} failures ---')
    t0 = time.time()
    for j, fid in enumerate(fails):
        c = state[fid]
        core = feedback_core(c['func_before'], c['patch'], c['compile_error'], c['cwe_id'])
        try:
            raw = generate_one(core)
        except RuntimeError as e:
            if 'out of memory' in str(e).lower():
                torch.cuda.empty_cache(); continue
            raise
        patch = extract_code(raw)
        r = is_compilable(patch)
        state[fid] = {**c, 'patch': patch, 'compilable': r['compilable'],
                      'compile_error': r['error']}
        if (j + 1) % 10 == 0:
            print(f'  {j+1}/{len(fails)}  ({(time.time()-t0)/60:.1f}m)')
    dump_round(rnd)
    nk = sum(c['compilable'] for c in state.values())
    print(f'round {rnd}: {nk}/{len(state)} compile ({nk/len(state)*100:.1f}%)')

print(f'\nDONE -> {OUTPUT_FILE}')
print('Download from Output tab; score change-aware similarity per round locally.')